# RPFR Data Exploration

Visualize per-atom RPFR values in 3D. Demonstrates two strategies for handling the cross-element scale problem.

## Setup

In [33]:
import sys
from pathlib import Path
import numpy as np
import h5py
import pandas as pd

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))

from rpfr_gui.data import ChemistryResolver, H5Provider
from rpfr_gui.ui import RPFRVisualizer

H5_PATH  = Path('/Users/simonandren/QM9_H5/qm9s.h5')
IDX_PATH = project_root / 'data/processed/index.parquet'

resolver = ChemistryResolver(IDX_PATH)
provider = H5Provider(H5_PATH)

print('Libraries loaded OK')

Libraries loaded OK


In [34]:
def load_molecule(smiles, temperature=300.0):
    """Resolve SMILES, load DFT coords and RPFR from HDF5."""
    mol_id = resolver.resolve(smiles, id_type='smiles')
    if mol_id is None:
        raise ValueError(f'Not found: {smiles}')
    with h5py.File(H5_PATH, 'r') as f:
        g = f[mol_id]
        symbols = [s.decode() for s in g['Atom_Symbol'][()]]
        coords  = np.stack([g['x'][()], g['y'][()], g['z'][()]], axis=1)
        rpfr    = g[f'RPFR_{int(temperature)}K'][()]
    return mol_id, symbols, coords, rpfr

## 1. Load a test molecule

**Methyl carbamate** (`COC(=O)NC(N)=O`) contains C, H, N, and O — ideal for demonstrating the cross-element scale problem.

In [35]:
SMILES = 'COC(=O)NC(N)=O'
TEMPERATURE = 300.0

mol_id, symbols, coords, rpfr = load_molecule(SMILES, TEMPERATURE)
print(f'Molecule ID: {mol_id}')

df = pd.DataFrame({'atom': symbols, 'x': coords[:,0], 'y': coords[:,1], 'z': coords[:,2], 'RPFR': rpfr})
print(f'{len(df)} atoms loaded\n')
print(df.groupby('atom')['RPFR'].describe().round(5))

Molecule ID: 005752
14 atoms loaded

      count      mean      std       min       25%       50%       75%  \
atom                                                                     
C       3.0   1.18068  0.03514   1.14014   1.16988   1.19962   1.20095   
H       6.0  14.17134  1.02189  13.19008  13.35903  13.92393  14.78134   
N       2.0   1.10352  0.01264   1.09458   1.09905   1.10352   1.10799   
O       3.0   1.12027  0.00573   1.11567   1.11707   1.11847   1.12258   

           max  
atom            
C      1.20229  
H     15.75225  
N      1.11246  
O      1.12669  


## The cross-element RPFR problem

RPFR values are **not comparable across elements**:

| Element | Typical RPFR at 300 K |
|---------|----------------------|
| H | 10 – 15 |
| C | 1.10 – 1.25 |
| N | 1.05 – 1.15 |
| O | 1.08 – 1.15 |

Hydrogen exchange involves a much larger mass ratio (²H/¹H ≈ 2) compared to heavy-atom isotopes (¹³C/¹²C ≈ 1.08), so H RPFR is physically on a different scale.

This notebook demonstrates two visualization strategies to handle this.

## 2. Build the visualizer

All visualizations use DFT-optimised 3D coordinates from the HDF5 file (not RDKit-generated conformers).

In [36]:
viz = RPFRVisualizer(SMILES, symbols, coords, rpfr, temperature=TEMPERATURE)
print(f'Molecule: {SMILES}')
print(f'Atoms: {viz.n_atoms}  |  Elements: {viz._elements}  |  T = {TEMPERATURE:.0f} K\n')
display(viz.summary_table().round(4))

Molecule: COC(=O)NC(N)=O
Atoms: 14  |  Elements: ['C', 'H', 'N', 'O']  |  T = 300 K



,idx,symbol,rpfr,el_mean,el_min,el_max,minmax
0,0,C,1.1401,1.1807,1.1401,1.2023,0.0000
1,1,O,1.1267,1.1203,1.1157,1.1267,1.0000
2,2,C,1.2023,1.1807,1.1401,1.2023,1.0000
3,3,O,1.1185,1.1203,1.1157,1.1267,0.2544
4,4,N,1.1125,1.1035,1.0946,1.1125,1.0000
5,5,C,1.1996,1.1807,1.1401,1.2023,0.9570
6,6,N,1.0946,1.1035,1.0946,1.1125,0.0000
7,7,O,1.1157,1.1203,1.1157,1.1267,0.0000
8,8,H,13.7154,14.1713,13.1901,15.7523,0.2050
9,9,H,13.1901,14.1713,13.1901,15.7523,0.0000


## 3. Mode 1 — Element filter

**Scientifically strictest.** Display only atoms of one element at a time, coloured on their own min-max RPFR scale. Other atoms shown as faint grey sticks for structural context.

In [37]:
viz.show_element_filter('H')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [38]:
viz.show_element_filter('C')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [39]:
viz.show_element_filter('O')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4. Mode 2 — Multi-element (independent colour scales)

All atoms shown simultaneously, each element with its own colormap:
- **H** → Blues, **C** → Greens, **N** → Purples, **O** → Oranges

Each element is independently min-max normalised.

**Warning:** Colours are NOT comparable across elements.

In [40]:
viz.show_multi_element()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5. Compare molecules: benzene vs methane

Symmetric molecules where all atoms of a given element are equivalent.

In [43]:
mol_id_b, syms_b, xyz_b, rpfr_b = load_molecule('c1ccccc1')
mol_id_m, syms_m, xyz_m, rpfr_m = load_molecule('C')

viz_benzene = RPFRVisualizer('c1ccccc1', syms_b, xyz_b, rpfr_b, temperature=TEMPERATURE)
viz_methane = RPFRVisualizer('C', syms_m, xyz_m, rpfr_m, temperature=TEMPERATURE)

print(f'Benzene ({mol_id_b}) — {len(syms_b)} atoms')
print(f'Methane ({mol_id_m}) — {len(syms_m)} atoms')

Benzene (000197) — 12 atoms
Methane (000001) — 5 atoms


In [44]:
print('Benzene — per-element minmax:')
viz_benzene.show_multi_element()

Benzene — per-element minmax:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [45]:
print('Methane — per-element minmax:')
viz_methane.show_multi_element()

Methane — per-element minmax:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Summary

| Mode | Best for |
|------|----------|
| `show_element_filter()` | Strict single-element analysis |
| `show_multi_element()` | Overview with per-element variation visible |

See **`02_rpfr_3d_visualization.ipynb`** for interactive molecule selector widget.